# Improving Query Process

## 1. Query Rewritting

Query Rewriting, is a query-time process. It happens after the user submits their query but before we perform the vector search. The goal is to transform the user's raw query into one or more new, optimized queries that are semantically richer or better structured for retrieval.

This is critical because a user's question and the document chunk that answers it often have a `semantic gap`—they don't share the same keywords or phrasing.

We will explore three key query-time techniques using `openai` and `qdrant_client`:

1. HyDE (Hypothetical Document Embeddings).

2. Query Decomposition: Breaking complex, multi-part questions into simpler sub-queries.

3. Step-Back Prompting: Finding a more general, high-level query to retrieve broader context.

In [35]:
import os
import json
import openai
from qdrant_client import QdrantClient, models
from sentence_transformers import SentenceTransformer
from dotenv import load_dotenv
from typing import List

# Load environment variables (ensure your .env file has OPENAI_API_KEY)
load_dotenv()

True

In [36]:
# 1. Initialize OpenAI Client
try:
    client = openai.OpenAI()
    CHAT_MODEL = "gpt-4.1-mini-2025-04-14" # Using a fast and effective model
    print(f"OpenAI client initialized. Using model: {CHAT_MODEL}")
except Exception as e:
    print(f"Error initializing OpenAI client: {e}")
    print("Please ensure your OPENAI_API_KEY is set in a .env file.")

# 2. Initialize Qdrant Client and Encoder
qdrant = QdrantClient(":memory:")
encoder = SentenceTransformer('all-MiniLM-L6-v2')
print("Qdrant client and SentenceTransformer encoder initialized.")

# 3. Define the collection we will search against
# This is the *simple* collection you create later
search_collection_name = "documents_only"

OpenAI client initialized. Using model: gpt-4.1-mini-2025-04-14
Qdrant client and SentenceTransformer encoder initialized.


In [37]:
# 4. Helper function to create and populate the collection for our tests
# We re-use the 'chunks' from your notebook
def setup_simple_collection(client, encoder, collection_name):
    try:
        client.recreate_collection(
            collection_name=collection_name,
            vectors_config=models.VectorParams(
                size=encoder.get_sentence_embedding_dimension(),
                distance=models.Distance.COSINE
            )
        )
        
        chunks = [
            "A mitochondrion (pl. mitochondria) is an organelle found in the cells of most eukaryotes... They use aerobic respiration to generate adenosine triphosphate (ATP), which is used throughout the cell as a source of chemical energy... The mitochondrion is popularly nicknamed the \"powerhouse of the cell\"...",
            "Python is a high-level, general-purpose programming language. Its design philosophy emphasizes code readability... Python is dynamically typed and garbage-collected. It supports multiple programming paradigms...",
            "The American Civil War (from April 12, 1861 to May 26, 1865) was a civil war in the United States between the Union... The central conflict leading to war was a dispute over whether slavery should be permitted to expand..."
        ]
        
        # This is the same logic as your cell
        client.upload_points(
            collection_name=collection_name,
            points=[
                models.PointStruct(
                    id=idx,
                    vector=encoder.encode(document).tolist(),
                    payload={"document": document}
                ) for idx, document in enumerate(chunks)
            ]
        )
        print(f"Collection '{collection_name}' created and populated with {len(chunks)} documents.")
    except Exception as e:
        print(f"Error setting up collection: {e}")

In [38]:
# 5. Helper function to search Qdrant
def search_qdrant(query: str, collection_name: str, limit: int = 1):
    result = qdrant.query_points(
        collection_name=collection_name,
        query=encoder.encode(query).tolist(),
        limit=limit
    )
    return result.points

In [39]:
# --- Run the setup ---
setup_simple_collection(qdrant, encoder, search_collection_name)

Collection 'documents_only' created and populated with 3 documents.


## Technique 1: HyDE (Hypothetical Document Embeddings)
This is the "classic" query-time version of HyDE. Instead of embedding the question, we first ask an LLM to generate a hypothetical answer. We then use the embedding of that hypothetical answer to perform the retrieval.

Why? The generated answer is semantically much closer to the actual documents in our database than the question is.

Let's test this with the query: `"What generates energy in a cell?"`.

In [40]:
# 1. Define our query
original_query_1 = "What generates energy in a cell?"
print(f"Original Query: {original_query_1}\n")

# 2. Create a prompt to generate a hypothetical document
hyde_prompt = f"""
Please write a brief, one-paragraph hypothetical document that perfectly answers 
the following question. This document will be used for a vector search.

Question: {original_query_1}

Hypothetical Document:
"""

Original Query: What generates energy in a cell?



In [41]:
response = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role": "user", "content": hyde_prompt}],
        temperature=0.0 # We want a factual, non-creative answer
    )
hypothetical_document = response.choices[0].message.content
print(f"--- Generated HyDE Document --- \n{hypothetical_document}\n")

--- Generated HyDE Document --- 
Energy in a cell is primarily generated by the mitochondria, often referred to as the "powerhouses" of the cell. These organelles convert nutrients, especially glucose, into adenosine triphosphate (ATP) through a process called cellular respiration. This process involves glycolysis, the citric acid cycle, and oxidative phosphorylation, where energy stored in chemical bonds is transformed into ATP, the main energy currency used to power various cellular activities.



In [42]:
print("--- 1. Search with ORIGINAL Query ---")
original_hits = search_qdrant(original_query_1, search_collection_name)
for hit in original_hits:
    print(f"Score: {hit.score:.4f}")
    print(f"Document: {hit.payload['document'][:70]}...\n")
    # This score matches your finding in cell

--- 1. Search with ORIGINAL Query ---
Score: 0.6064
Document: A mitochondrion (pl. mitochondria) is an organelle found in the cells ...



In [43]:
# 5. Search using the HyDE document
print("--- 2. Search with HyDE Document ---")
hyde_hits = search_qdrant(hypothetical_document, search_collection_name)
for hit in hyde_hits:
    print(f"Score: {hit.score:.4f}")
    print(f"Document: {hit.payload['document'][:70]}...\n")

--- 2. Search with HyDE Document ---
Score: 0.7948
Document: A mitochondrion (pl. mitochondria) is an organelle found in the cells ...



## Technique 2: Query Decomposition (Sub-Queries)
This technique is for complex, multi-part questions. A single vector embedding often can't represent multiple, distinct ideas. We use an LLM to break the complex query into multiple simpler, self-contained sub-queries. We then run retrieval for each sub-query and combine the results.

Query: `"What is Python and what was the cause of the American Civil War?"`

In [44]:
# 1. Define our complex query
original_query_2 = "What is Python and what was the cause of the American Civil War?"
print(f"Original Query: {original_query_2}\n")

# 2. Create a prompt to decompose the query into JSON
decomposition_prompt = f"""
Decompose the following complex user question into a list of 
simple, self-contained sub-queries. Return them as a JSON object with a single key "queries".

Question: {original_query_2}
"""

Original Query: What is Python and what was the cause of the American Civil War?



In [45]:
response = client.chat.completions.create(
    model=CHAT_MODEL,
    response_format={"type": "json_object"},
    messages=[{"role": "user", "content": decomposition_prompt}]
)

sub_queries = json.loads(response.choices[0].message.content)["queries"]
print(f"--- Generated Sub-Queries --- \n{sub_queries}\n")

--- Generated Sub-Queries --- 
['What is Python?', 'What was the cause of the American Civil War?']



In [46]:
# 3. Retrieve documents for EACH sub-query
all_docs = {} # Use a dict to de-duplicate based on payload
print("--- 1. Search with Sub-Queries ---")
for q in sub_queries:
    print(f"Searching for: '{q}'")
    sub_hits = search_qdrant(q, search_collection_name)
    for hit in sub_hits:
        # Use doc as key to de-duplicate
        all_docs[hit.payload['document']] = hit.score

print("\n--- Combined Results from Sub-Queries ---")
for doc, score in all_docs.items():
        print(f"Score: {score:.4f}, Doc: {doc[:70]}...")

--- 1. Search with Sub-Queries ---
Searching for: 'What is Python?'
Searching for: 'What was the cause of the American Civil War?'

--- Combined Results from Sub-Queries ---
Score: 0.8273, Doc: Python is a high-level, general-purpose programming language. Its desi...
Score: 0.7208, Doc: The American Civil War (from April 12, 1861 to May 26, 1865) was a civ...


In [47]:
# 4. Compare with searching for the ORIGINAL query
print("\n--- 2. Search with ORIGINAL (Combined) Query ---")
original_hits = search_qdrant(original_query_2, search_collection_name, limit=2)
for hit in original_hits:
    print(f"Score: {hit.score:.4f}, Doc: {hit.payload['document'][:70]}...")


--- 2. Search with ORIGINAL (Combined) Query ---
Score: 0.4940, Doc: The American Civil War (from April 12, 1861 to May 26, 1865) was a civ...
Score: 0.4482, Doc: Python is a high-level, general-purpose programming language. Its desi...


## Technique 3: Step-Back Prompting
This technique helps when a user's query is very specific or "lost in the weeds." We ask an LLM to "step back" and generate a more general, high-level question. We then retrieve documents for both the original specific query and the new general query. This provides high-level context and specific details.

Query: `"Who was Carl Benda?"` (A specific name mentioned in the mitochondria document)

In [48]:
# 1. Define our specific query
original_query_3 = "Who was Carl Benda?"
print(f"Original Query: {original_query_3}\n")

# 2. Create a prompt to generate a "step-back" question
step_back_prompt = f"""
You will be given a specific question. Your task is to generate a more general,
high-level "step-back question" that helps provide context to answer the original.

Question: {original_query_3}

Step-back question:
"""

Original Query: Who was Carl Benda?



In [49]:
response = client.chat.completions.create(
    model=CHAT_MODEL,
    messages=[{"role": "user", "content": step_back_prompt}],
    temperature=0.0
)
step_back_query = response.choices[0].message.content.strip()
print(f"--- Generated Step-Back Query --- \n{step_back_query}\n")

--- Generated Step-Back Query --- 
What are the key contributions and historical significance of individuals named Carl Benda?



In [50]:
print("--- 1. Search with ORIGINAL Query ---")
original_hits = search_qdrant(original_query_3, search_collection_name)
for hit in original_hits:
    print(f"Score: {hit.score:.4f}, Doc: {hit.payload['document'][:70]}...")

--- 1. Search with ORIGINAL Query ---
Score: 0.0360, Doc: The American Civil War (from April 12, 1861 to May 26, 1865) was a civ...


In [51]:
print("\n--- 2. Search with STEP-BACK Query ---")
step_back_hits = search_qdrant(step_back_query, search_collection_name)
for hit in step_back_hits:
    print(f"Score: {hit.score:.4f}, Doc: {hit.payload['document'][:70]}...")


--- 2. Search with STEP-BACK Query ---
Score: 0.0368, Doc: The American Civil War (from April 12, 1861 to May 26, 1865) was a civ...


In [52]:
# 4. Combine results
combined_docs = {hit.payload['document'] for hit in original_hits + step_back_hits}
combined_docs

{'The American Civil War (from April 12, 1861 to May 26, 1865) was a civil war in the United States between the Union... The central conflict leading to war was a dispute over whether slavery should be permitted to expand...'}

## 2. Query Expansion 

### Improve Semantic Similarity with Reverse HYDE

It is common that the documents that we want to retrieve are longer than the users' queries and have different formats. To increase the accuracy of the **retrieval** of the documents based on the users' queries, we will generate hypothetical potential queries from each document and use them as vector embeddings to the documents - AKA Reverse Hyde.

Please note that the original [Hyde technique](https://arxiv.org/abs/2212.10496) processed the incoming queries of the users, and generated the hypothetical documents from these queries, and then used these hypothetical documents to retrive the real documents. In the reverse HYDE, the processing is done when indexing the documents and not in retrival time. Therefore, the latency of the query is not affectd.

* [Reverse Hyde Implementation](#reverse-hyde-implementation)
* [Enriching Vector Database with Reverse Hyde Output](#enriching-vector-database-with-reverse-hyde-output)
* [Query the Enriched Index](#query-the-enriched-index)

In [53]:
import warnings

# Suppress warnings
warnings.filterwarnings('ignore')

### Reverse HYDE Implementation

We will create a class that will generate and hypotherical questions and also retrieve the document by calculating the semantic similarity matching. In a real application, we can use a vector database for the embedding vector storage, indexing and retrieval. 

In [54]:
import openai
from typing import List, Dict
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

In [55]:
class ReverseHyde:
    def __init__(self, api_key: str):
        openai.api_key = api_key
        self.chat_model = "gpt-4.1-mini-2025-04-14"

    def generate_reverse_hyde(self, chunk: str, n: int = 3) -> List[str]:
        prompt = f"""
        
Given the following text chunk, generate {n} different questions that this chunk would be a good answer to:

Chunk: {chunk}

Questions (enumarate the questions with 1. 2., etc.):
"""

        client = openai.OpenAI()
        response = client.chat.completions.create(
            model=self.chat_model,
            messages=[
                {"role": "system", "content": "You are a helpful assistant."},
                {"role": "user", "content": prompt}
            ],
            max_tokens=100,
            n=1,
            stop=None,
            temperature=0.7,
        )

        
        questions = response.choices[0].message.content.strip().split('\n')
        return [q.split('. ', 1)[1] for q in questions if '. ' in q]

    def process_chunks(self, chunks: List[str], n: int = 3) -> Dict[str, List[str]]:
        processed_chunks = {}
        for chunk in chunks:
            processed_chunks[chunk] = self.generate_reverse_hyde(chunk, n)
        return processed_chunks


Loading API keys from environment variable

In [56]:
from dotenv import load_dotenv

load_dotenv()

True

### Enriching Vector Database with Reverse Hyde Output

We will apply the Reverse Hyde method on a set of documents, and enrich the vector database index with LLM generated Hypothetical questions.

In [57]:
import os
# Usage example
api_key = os.getenv("OPENAI_API_KEY")
reverse_hyde = ReverseHyde(api_key)

chunks = [
    "A mitochondrion (pl. mitochondria) is an organelle found in the cells of most eukaryotes, such as animals, plants and fungi. Mitochondria have a double membrane structure and use aerobic respiration to generate adenosine triphosphate (ATP), which is used throughout the cell as a source of chemical energy. They were discovered by Albert von Kölliker in 1857 in the voluntary muscles of insects. Meaning a thread-like granule, the term mitochondrion was coined by Carl Benda in 1898. The mitochondrion is popularly nicknamed the \"powerhouse of the cell\", a phrase popularized by Philip Siekevitz in a 1957 Scientific American article of the same name.",
    "Python is a high-level, general-purpose programming language. Its design philosophy emphasizes code readability with the use of significant indentation. Python is dynamically typed and garbage-collected. It supports multiple programming paradigms, including structured (particularly procedural), object-oriented and functional programming. It is often described as a \"batteries included\" language due to its comprehensive standard library.",
    "The American Civil War (from April 12, 1861 to May 26, 1865) was a civil war in the United States between the Union (\"the North\") and the Confederacy (\"the South\"), which was formed in 1861 by states that had seceded from the Union. The central conflict leading to war was a dispute over whether slavery should be permitted to expand into the western territories, leading to more slave states, or be prohibited from doing so, which many believed would place slavery on a course of ultimate extinction."
]

processed_chunks = reverse_hyde.process_chunks(chunks, n=5)

In [58]:
import json

print(json.dumps(processed_chunks, indent=4))

{
    "A mitochondrion (pl. mitochondria) is an organelle found in the cells of most eukaryotes, such as animals, plants and fungi. Mitochondria have a double membrane structure and use aerobic respiration to generate adenosine triphosphate (ATP), which is used throughout the cell as a source of chemical energy. They were discovered by Albert von K\u00f6lliker in 1857 in the voluntary muscles of insects. Meaning a thread-like granule, the term mitochondrion was coined by Carl Benda in 1898. The mitochondrion is popularly nicknamed the \"powerhouse of the cell\", a phrase popularized by Philip Siekevitz in a 1957 Scientific American article of the same name.": [
        "What is a mitochondrion and in which types of cells is it found?  ",
        "How do mitochondria generate energy for the cell?  ",
        "Who discovered mitochondria and when were they discovered?  ",
        "What is the origin and meaning of the term \"mitochondrion\"?  ",
        "Why is the mitochondrion called t

### Query the enriched index

Once we have an index with multiple hypothetical questions to the documents, we can use it to retrive the document based on a real user's query.

In [59]:
query = "What generates energy in a cell?"

In [60]:
from qdrant_client import models, QdrantClient
from sentence_transformers import SentenceTransformer

# create the vector database client
qdrant = QdrantClient(":memory:") # Create in-memory Qdrant instance

# Create the embedding encoder
encoder = SentenceTransformer('all-MiniLM-L6-v2') # Model to create embeddings

In [61]:
# Create collection to store the wine rating data
hyde_collection_name="reverse_hyde"

qdrant.recreate_collection(
    collection_name=hyde_collection_name,
    vectors_config=models.VectorParams(
        size=encoder.get_sentence_embedding_dimension(), # Vector size is defined by used model
        distance=models.Distance.COSINE
    )
)

True

In [62]:
import uuid
# vectorize!
qdrant.upload_points(
    collection_name=hyde_collection_name,
    points=[
        models.PointStruct(
            id=uuid.uuid5(uuid.NAMESPACE_URL, f"{d_idx}-{q_idx}").hex,
            vector=encoder.encode(question).tolist(),
            payload={ 
                "document": document , 
                "doc_id": d_idx
            }
        ) for d_idx, (document, questions) 
            in enumerate(processed_chunks.items()) 
                for q_idx, question in enumerate(questions)
    ]
)

In [63]:
print(
    qdrant
    .get_collection(
        collection_name=hyde_collection_name
    )
)

status=<CollectionStatus.GREEN: 'green'> optimizer_status=<OptimizersStatusOneOf.OK: 'ok'> warnings=None indexed_vectors_count=0 points_count=15 segments_count=1 config=CollectionConfig(params=CollectionParams(vectors=VectorParams(size=384, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, datatype=None, multivector_config=None), shard_number=None, sharding_method=None, replication_factor=None, write_consistency_factor=None, read_fan_out_factor=None, on_disk_payload=None, sparse_vectors=None), hnsw_config=HnswConfig(m=16, ef_construct=100, full_scan_threshold=10000, max_indexing_threads=0, on_disk=None, payload_m=None, inline_storage=None), optimizer_config=OptimizersConfig(deleted_threshold=0.2, vacuum_min_vector_number=1000, default_segment_number=0, max_segment_size=None, memmap_threshold=None, indexing_threshold=20000, flush_interval_sec=5, max_optimization_threads=1), wal_config=WalConfig(wal_capacity_mb=32, wal_segments_ahead=0, wal_r

### Search Collection for Best Match



In [64]:
def search_collection(collection_name: str, query: str, limit: int = 1):
    """
    This function searches the specified collection for the best match to the given query.
    It then creates a table and a panel to display the query and the best match.
    
    :param collection_name: The name of the collection to search.
    :param query: The query to search for.
    :param limit: The maximum number of results to return. Default is 1.
    """
    result = qdrant.query_points(
        collection_name=collection_name,
        query=encoder.encode(query).tolist(),
        limit=limit
    )
    hits = result.points

    
    print("Query: ", query)
    print("Best Matching Chunk: ", hits[0].payload['document'])
    print("Score: ", hits[0].score)


In [65]:
search_collection(hyde_collection_name, query)

Query:  What generates energy in a cell?
Best Matching Chunk:  A mitochondrion (pl. mitochondria) is an organelle found in the cells of most eukaryotes, such as animals, plants and fungi. Mitochondria have a double membrane structure and use aerobic respiration to generate adenosine triphosphate (ATP), which is used throughout the cell as a source of chemical energy. They were discovered by Albert von Kölliker in 1857 in the voluntary muscles of insects. Meaning a thread-like granule, the term mitochondrion was coined by Carl Benda in 1898. The mitochondrion is popularly nicknamed the "powerhouse of the cell", a phrase popularized by Philip Siekevitz in a 1957 Scientific American article of the same name.
Score:  0.8280217009077394


### Compare to document only index (without Hyde)

We will index the same documents without adding the reverse Hyde questions and compare the similarity scores.

In [66]:
# Create collection to store the wine rating data
docs_collection_name="documents_only"

qdrant.recreate_collection(
    collection_name=docs_collection_name,
    vectors_config=models.VectorParams(
        size=encoder.get_sentence_embedding_dimension(), # Vector size is defined by used model
        distance=models.Distance.COSINE
    )
)

True

In [67]:
# vectorize!
qdrant.upload_points(
    collection_name=docs_collection_name,
    points=[
        models.PointStruct(
            id=idx,
            vector=encoder.encode(document).tolist(),
            payload={ "document": document}
        ) for idx, (document, questions) in enumerate(processed_chunks.items())
    ]
)

In [68]:
search_collection(docs_collection_name, query)

Query:  What generates energy in a cell?
Best Matching Chunk:  A mitochondrion (pl. mitochondria) is an organelle found in the cells of most eukaryotes, such as animals, plants and fungi. Mitochondria have a double membrane structure and use aerobic respiration to generate adenosine triphosphate (ATP), which is used throughout the cell as a source of chemical energy. They were discovered by Albert von Kölliker in 1857 in the voluntary muscles of insects. Meaning a thread-like granule, the term mitochondrion was coined by Carl Benda in 1898. The mitochondrion is popularly nicknamed the "powerhouse of the cell", a phrase popularized by Philip Siekevitz in a 1957 Scientific American article of the same name.
Score:  0.5002659397930831
